# SAE + POD + SINDy — baseado na técnica do artigo

Pipeline para identificação SINDy sobre coordenadas latentes SAE/POD, com reconstrução dos campos físicos e comparação visual/métrica contra dados OpenFOAM.

Etapas principais:

1. Carregamento dos latentes SAE.
2. Transformação POD no espaço latente.
3. Biblioteca SINDy com termos polinomiais, temporais e periódicos opcionais.
4. Regressão esparsa por Adaptive Lasso ou TLSA/STLSQ.
5. Seleção automática por erro de trajetória, erro de derivadas e AIC.
6. Reconstrução dos campos físicos por decoder SAE.
7. Comparação contra OpenFOAM bruto interpolado.


## 1. Importações, caminhos e utilidades


In [1]:
from __future__ import annotations

import csv
import json
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp
from scipy.interpolate import griddata
from scipy.ndimage import zoom
from scipy.signal import savgol_filter
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Lasso

plt.switch_backend("Agg")
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SRC_DIR = Path(r"/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SINDy/src").resolve()
CASE_DIR = SRC_DIR.parent.parent
OUTPUT_ROOT = CASE_DIR / "SINDy" / "resultados" / "sae_pod_sindy"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"SRC_DIR     = {SRC_DIR}")
print(f"CASE_DIR    = {CASE_DIR}")
print(f"OUTPUT_ROOT = {OUTPUT_ROOT}")


def force_matplotlib_agg() -> None:
    matplotlib.use("Agg", force=True)
    plt.switch_backend("Agg")


def relative_l2(reference: np.ndarray, estimate: np.ndarray) -> float:
    return float(np.linalg.norm(reference - estimate) / (np.linalg.norm(reference) + 1e-12))


def make_json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {key: make_json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


SRC_DIR     = /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SINDy/src
CASE_DIR    = /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))
OUTPUT_ROOT = /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SINDy/resultados/sae_pod_sindy_article_copy


## 2. Carregar latentes SAE e aplicar POD no espaço latente


In [2]:
@dataclass
class ReducedDataset:
    name: str
    state_symbol: str
    states: np.ndarray
    times: np.ndarray
    source: Path
    metadata: dict[str, Any] = field(default_factory=dict)


@dataclass
class LatentPODTransform:
    z_mean: np.ndarray
    z_std: np.ndarray
    pod_mean: np.ndarray
    components: np.ndarray
    singular_values: np.ndarray
    energy_capture: float
    q_mean: np.ndarray
    q_std: np.ndarray

    def transform(self, z: np.ndarray) -> np.ndarray:
        z = np.asarray(z, dtype=np.float64)
        z_norm = (z - self.z_mean[None, :]) / self.z_std[None, :]
        q = (z_norm - self.pod_mean[None, :]) @ self.components.T
        return (q - self.q_mean[None, :]) / self.q_std[None, :]

    def inverse_transform(self, q_scaled: np.ndarray) -> np.ndarray:
        q_scaled = np.asarray(q_scaled, dtype=np.float64)
        q = q_scaled * self.q_std[None, :] + self.q_mean[None, :]
        z_norm = q @ self.components + self.pod_mean[None, :]
        return z_norm * self.z_std[None, :] + self.z_mean[None, :]


def sort_and_deduplicate(times: np.ndarray, states: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    order = np.argsort(times, kind="stable")
    times = np.asarray(times[order], dtype=np.float64)
    states = np.asarray(states[order], dtype=np.float64)
    unique_times, unique_idx = np.unique(times, return_index=True)
    if len(unique_times) != len(times):
        times = times[unique_idx]
        states = states[unique_idx]
    return times, states


def infer_dt(times: np.ndarray) -> float:
    deltas = np.diff(times)
    return 0.0 if deltas.size == 0 else float(np.median(deltas))


def load_sae_latents(case_dir: Path) -> ReducedDataset:
    candidates = (
        Path(case_dir) / "SAE" / "resultados" / "latents.npz",
        Path(case_dir) / "SAE" / "resultados" / "SAE" / "latents.npz",
    )
    latent_path = next((path for path in candidates if path.exists()), None)
    if latent_path is None:
        checked = "\n".join(str(path) for path in candidates)
        raise FileNotFoundError(f"Latentes SAE não encontrados. Caminhos testados:\n{checked}")

    data = np.load(latent_path, allow_pickle=True)
    train_end_time = None
    if {"z_train", "z_val", "t_train", "t_val"}.issubset(set(data.files)):
        z_train = np.asarray(data["z_train"], dtype=np.float64)
        z_val = np.asarray(data["z_val"], dtype=np.float64)
        t_train = np.asarray(data["t_train"], dtype=np.float64)
        t_val = np.asarray(data["t_val"], dtype=np.float64)
        states = np.vstack([z_train, z_val])
        times = np.concatenate([t_train, t_val])
        train_end_time = float(np.max(t_train)) if t_train.size else None
    elif "latent_data" in data.files:
        states = np.asarray(data["latent_data"], dtype=np.float64)
        times = np.asarray(data["times"], dtype=np.float64) if "times" in data.files else np.arange(states.shape[0], dtype=np.float64)
    else:
        raise KeyError("Formato de latents.npz não suportado.")

    times, states = sort_and_deduplicate(times, states)
    train_count = int(np.searchsorted(times, train_end_time, side="right")) if train_end_time is not None else int(0.8 * len(times))
    metadata = {"latent_dim": int(states.shape[1]), "dt": infer_dt(times), "train_count": train_count}
    return ReducedDataset("sae_latent", "z", states, times, latent_path, metadata)


def fit_latent_pod(dataset: ReducedDataset, n_modes: int) -> tuple[ReducedDataset, LatentPODTransform]:
    z = np.asarray(dataset.states, dtype=np.float64)
    z_mean = np.mean(z, axis=0)
    z_std = np.std(z, axis=0)
    z_std[z_std < 1e-12] = 1.0
    z_norm = (z - z_mean[None, :]) / z_std[None, :]

    pod_mean = np.mean(z_norm, axis=0)
    z_centered = z_norm - pod_mean[None, :]
    U, s, Vt = np.linalg.svd(z_centered, full_matrices=False)

    r = min(max(1, int(n_modes)), Vt.shape[0])
    components = Vt[:r, :]
    q = z_centered @ components.T
    energy_capture = float(np.sum(s[:r] ** 2) / (np.sum(s**2) + 1e-12))

    q_mean = np.mean(q, axis=0)
    q_std = np.std(q, axis=0)
    q_std[q_std < 1e-12] = 1.0
    q_scaled = (q - q_mean[None, :]) / q_std[None, :]

    transform = LatentPODTransform(z_mean, z_std, pod_mean, components, s, energy_capture, q_mean, q_std)
    reduced = ReducedDataset(
        "sae_pod_latent",
        "r",
        q_scaled,
        dataset.times,
        dataset.source,
        {
            "latent_dim_original": int(z.shape[1]),
            "latent_pod_modes": int(r),
            "latent_pod_energy_capture": energy_capture,
            "dt": infer_dt(dataset.times),
            "train_count": int(dataset.metadata.get("train_count", int(0.8 * len(dataset.times)))),
        },
    )
    return reduced, transform


def smooth_states(states: np.ndarray, window_length: int) -> np.ndarray:
    if window_length is None or window_length <= 2:
        return np.asarray(states, dtype=np.float64)
    window = int(window_length)
    if window % 2 == 0:
        window += 1
    if window >= states.shape[0]:
        window = states.shape[0] - 1 if states.shape[0] % 2 == 0 else states.shape[0]
    if window < 5:
        return np.asarray(states, dtype=np.float64)
    return savgol_filter(states, window_length=window, polyorder=3, axis=0)


## 3. Biblioteca SINDy: polinômios, tempo e sen/cos temporal


In [3]:
def generate_powers(n_variables: int, order: int) -> list[tuple[int, ...]]:
    if n_variables == 1:
        return [(power,) for power in range(order + 1)]
    powers = []
    for power in range(order + 1):
        for suffix in generate_powers(n_variables - 1, order - power):
            powers.append((power,) + suffix)
    return sorted(powers, reverse=False)


def estimate_frequency(times: np.ndarray, signal: np.ndarray) -> float:
    times = np.asarray(times, dtype=np.float64)
    signal = np.asarray(signal, dtype=np.float64) - np.mean(signal)
    dt = infer_dt(times)
    if dt <= 0 or len(signal) < 8:
        return 1.0
    freq = np.fft.rfftfreq(len(signal), d=dt)
    amp = np.abs(np.fft.rfft(signal))
    if len(freq) <= 1:
        return 1.0
    idx = int(np.argmax(amp[1:]) + 1)
    return float(freq[idx]) if np.isfinite(freq[idx]) and freq[idx] > 0 else 1.0


@dataclass
class SindyLibrary:
    n_variables: int
    polynomial_order: int = 3
    use_state_sine: bool = False
    use_time_poly: bool = True
    use_time_sine: bool = False
    time_order: int = 3
    frequency: float = 1.0
    variable_symbol: str = "r"

    def __post_init__(self) -> None:
        self.powers = generate_powers(self.n_variables, self.polynomial_order)
        self.feature_names = self._build_feature_names()

    def _build_feature_names(self) -> list[str]:
        names = []
        for power in self.powers:
            if sum(power) == 0:
                names.append("1")
                continue
            parts = []
            for index, exponent in enumerate(power):
                if exponent == 0:
                    continue
                variable = f"{self.variable_symbol}{index + 1}"
                parts.append(variable if exponent == 1 else f"{variable}^{exponent}")
            names.append("*".join(parts))
        if self.use_state_sine:
            for index in range(self.n_variables):
                variable = f"{self.variable_symbol}{index + 1}"
                names += [f"sin({variable})", f"cos({variable})"]
        if self.use_time_poly:
            for p in range(1, self.time_order + 1):
                names.append("t" if p == 1 else f"t^{p}")
        if self.use_time_sine:
            names += [f"sin(2π*{self.frequency:.6g}*t)", f"cos(2π*{self.frequency:.6g}*t)"]
        return names

    def transform(self, states: np.ndarray, times: np.ndarray | float) -> np.ndarray:
        states = np.atleast_2d(np.asarray(states, dtype=np.float64))
        if np.isscalar(times):
            t = np.full(states.shape[0], float(times), dtype=np.float64)
        else:
            t = np.asarray(times, dtype=np.float64).ravel()
            if t.size == 1 and states.shape[0] > 1:
                t = np.full(states.shape[0], float(t[0]), dtype=np.float64)
        theta = np.empty((states.shape[0], len(self.feature_names)), dtype=np.float64)
        col = 0
        for power in self.powers:
            theta[:, col] = np.prod(np.power(states, power), axis=1)
            col += 1
        if self.use_state_sine:
            for index in range(self.n_variables):
                theta[:, col] = np.sin(states[:, index])
                theta[:, col + 1] = np.cos(states[:, index])
                col += 2
        if self.use_time_poly:
            t0 = float(np.min(t)) if t.size > 1 else 0.0
            tmax = float(np.max(t)) if t.size > 1 else 1.0
            denom = max(tmax - t0, 1e-12)
            tn = (t - t0) / denom if t.size > 1 else np.asarray(t, dtype=np.float64)
            for p in range(1, self.time_order + 1):
                theta[:, col] = tn**p
                col += 1
        if self.use_time_sine:
            theta[:, col] = np.sin(2 * np.pi * self.frequency * t)
            theta[:, col + 1] = np.cos(2 * np.pi * self.frequency * t)
            col += 2
        return theta

    def transform_single(self, state: np.ndarray, time: float) -> np.ndarray:
        return self.transform(np.asarray(state, dtype=np.float64)[None, :], float(time))[0]


def compute_derivatives(states: np.ndarray, times: np.ndarray) -> np.ndarray:
    return np.gradient(states, times, axis=0, edge_order=2 if len(times) > 2 else 1)


## 4. Regressões esparsas: Adaptive Lasso e TLSA/STLSQ com ridge


In [4]:
def fit_adaptive_lasso(theta: np.ndarray, target: np.ndarray, alpha: float, delta: float = 3.0, max_adapt_iter: int = 100) -> np.ndarray:
    n_features = theta.shape[1]
    weights = np.ones(n_features, dtype=np.float64)
    coef = np.zeros(n_features, dtype=np.float64)
    for _ in range(max_adapt_iter):
        prev = weights.copy()
        theta_w = theta / weights[None, :]
        model = Lasso(alpha=alpha, fit_intercept=False, max_iter=50000, tol=1e-6)
        model.fit(theta_w, target)
        coef = model.coef_ / weights
        weights = 1.0 / (np.abs(coef) ** delta + 1e-30)
        if np.mean((weights - prev) ** 2) < 1e-10:
            break
    return coef


def fit_tlsa_norm(theta: np.ndarray, target: np.ndarray, threshold: float, ridge: float = 1e-2, max_iter: int = 100) -> np.ndarray:
    n, d = theta.shape
    if ridge != 0:
        w = np.linalg.lstsq(theta.T @ theta + ridge * np.eye(d), theta.T @ target, rcond=None)[0]
    else:
        w = np.linalg.lstsq(theta, target, rcond=None)[0]
    relevant = d
    for _ in range(max_iter):
        small = np.where(np.abs(w) <= threshold)[0]
        big = np.array([i for i in range(d) if i not in small], dtype=int)
        if relevant == len(big):
            break
        relevant = len(big)
        if len(big) == 0:
            w[:] = 0.0
            break
        w[small] = 0.0
        if ridge != 0:
            w[big] = np.linalg.lstsq(theta[:, big].T @ theta[:, big] + ridge * np.eye(len(big)), theta[:, big].T @ target, rcond=None)[0]
        else:
            w[big] = np.linalg.lstsq(theta[:, big], target, rcond=None)[0]
    return w


def fit_sparse_coefficients(theta: np.ndarray, derivatives: np.ndarray, method: str, alpha: float) -> np.ndarray:
    Xi = np.zeros((theta.shape[1], derivatives.shape[1]), dtype=np.float64)
    for j in range(derivatives.shape[1]):
        if method == "alasso":
            Xi[:, j] = fit_adaptive_lasso(theta, derivatives[:, j], alpha)
        elif method == "tlsa_norm":
            Xi[:, j] = fit_tlsa_norm(theta, derivatives[:, j], threshold=alpha, ridge=1e-2)
        else:
            raise ValueError(f"Método desconhecido: {method}")
    return Xi


def simulate_sindy(library: SindyLibrary, coefficients: np.ndarray, times: np.ndarray, initial_state: np.ndarray, method: str = "RK45") -> tuple[np.ndarray | None, str | None]:
    def rhs(time: float, state: np.ndarray) -> np.ndarray:
        return library.transform_single(state, time) @ coefficients
    try:
        sol = solve_ivp(rhs, (float(times[0]), float(times[-1])), np.asarray(initial_state, dtype=np.float64), t_eval=times, method=method, rtol=1e-6, atol=1e-8)
    except Exception as exc:
        return None, str(exc)
    if not sol.success or sol.y.shape[1] != len(times):
        return None, str(sol.message)
    if not np.all(np.isfinite(sol.y)):
        return None, "Integração gerou NaN/Inf."
    return sol.y.T, None


def simulate_one_step(library: SindyLibrary, coefficients: np.ndarray, times: np.ndarray, reference_states: np.ndarray) -> np.ndarray:
    pred = np.zeros_like(reference_states, dtype=np.float64)
    pred[0] = reference_states[0]
    for i in range(1, len(times)):
        dt = float(times[i] - times[i - 1])
        rhs = library.transform_single(reference_states[i - 1], times[i - 1]) @ coefficients
        pred[i] = reference_states[i - 1] + dt * rhs
    return pred


def aic_score(reference: np.ndarray, estimate: np.ndarray | None, n_terms: int) -> float:
    if estimate is None:
        return 1e12
    rss = float(np.sum((reference - estimate) ** 2))
    n = int(reference.size)
    return float(n * np.log(rss / max(n, 1) + 1e-30) + 2 * n_terms)




def r2_score_array(reference: np.ndarray, estimate: np.ndarray) -> float:
    reference = np.asarray(reference, dtype=np.float64)
    estimate = np.asarray(estimate, dtype=np.float64)
    ss_res = float(np.sum((reference - estimate) ** 2))
    ss_tot = float(np.sum((reference - np.mean(reference)) ** 2))
    return float(1.0 - ss_res / (ss_tot + 1e-12))


def error_metrics(reference: np.ndarray, estimate: np.ndarray | None, prefix: str) -> dict[str, float | None]:
    if estimate is None or not np.all(np.isfinite(estimate)):
        return {
            f"{prefix}_rmse": None,
            f"{prefix}_mae": None,
            f"{prefix}_max_abs": None,
            f"{prefix}_rel_l2": None,
            f"{prefix}_r2": None,
        }
    diff = np.asarray(estimate, dtype=np.float64) - np.asarray(reference, dtype=np.float64)
    return {
        f"{prefix}_rmse": float(np.sqrt(np.mean(diff**2))),
        f"{prefix}_mae": float(np.mean(np.abs(diff))),
        f"{prefix}_max_abs": float(np.max(np.abs(diff))),
        f"{prefix}_rel_l2": relative_l2(reference, estimate),
        f"{prefix}_r2": r2_score_array(reference, estimate),
    }


def format_equations(coefficients: np.ndarray, names: list[str], symbol: str) -> str:
    lines = []
    for j in range(coefficients.shape[1]):
        parts = []
        for value, name in zip(coefficients[:, j], names):
            if abs(value) <= 1e-10:
                continue
            parts.append(f"{value:+.6e}" if name == "1" else f"{value:+.6e}*{name}")
        lines.append(f"d{symbol}{j+1}/dt = {' '.join(parts) if parts else '0.0'}")
    return "\n".join(lines)


## 5. Busca automática SINDy


In [5]:
@dataclass
class SearchConfig:
    latent_pod_modes_list: tuple[int, ...] = (4,)
    polynomial_orders: tuple[int, ...] = (1,)
    methods: tuple[str, ...] = ("alasso", "tlsa_norm")
    alpha_values: tuple[float, ...] = (1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1)
    smooth_windows: tuple[int, ...] = (0, 21, 51, 101)
    use_time_poly_options: tuple[bool, ...] = (True,)
    use_time_sine_options: tuple[bool, ...] = (False, True)
    use_state_sine_options: tuple[bool, ...] = (False,)
    integrator_methods: tuple[str, ...] = ("RK45", "Radau")
    trajectory_error_limit: float = 0.60
    derivative_error_limit: float = 0.90


def evaluate_candidate(latent_dataset: ReducedDataset, cfg: dict[str, Any]) -> dict[str, Any]:
    reduced, transform = fit_latent_pod(latent_dataset, cfg["latent_pod_modes"])
    states_fit = smooth_states(reduced.states, cfg["smooth_window"])
    derivatives = compute_derivatives(states_fit, reduced.times)
    freq = estimate_frequency(reduced.times, states_fit[:, 0])
    library = SindyLibrary(
        n_variables=reduced.states.shape[1],
        polynomial_order=cfg["polynomial_order"],
        use_state_sine=cfg["use_state_sine"],
        use_time_poly=cfg["use_time_poly"],
        use_time_sine=cfg["use_time_sine"],
        time_order=3,
        frequency=freq,
        variable_symbol=reduced.state_symbol,
    )
    theta = library.transform(states_fit, reduced.times)
    Xi = fit_sparse_coefficients(theta, derivatives, cfg["method"], cfg["alpha"])
    derivatives_pred = theta @ Xi
    derivative_error = relative_l2(derivatives, derivatives_pred)
    one_step = simulate_one_step(library, Xi, reduced.times, reduced.states)
    one_step_error = relative_l2(reduced.states, one_step)

    free_pred, message = simulate_sindy(library, Xi, reduced.times, reduced.states[0], method=cfg["integrator"])
    free_success = free_pred is not None
    free_error = None if free_pred is None else relative_l2(reduced.states, free_pred)
    n_nonzero = int(np.count_nonzero(np.abs(Xi) > 1e-10))
    n_possible = int(Xi.size)
    aic = aic_score(reduced.states, free_pred, n_nonzero)
    train_count = int(reduced.metadata.get("train_count", int(0.8 * len(reduced.times))))
    train_slice = slice(0, train_count)
    val_slice = slice(train_count, len(reduced.times))

    metrics = {}
    metrics.update(error_metrics(derivatives, derivatives_pred, "derivative"))
    metrics.update(error_metrics(reduced.states, one_step, "one_step"))
    metrics.update(error_metrics(reduced.states, free_pred, "free"))
    metrics.update(error_metrics(derivatives[train_slice], derivatives_pred[train_slice], "train_derivative"))
    metrics.update(error_metrics(reduced.states[train_slice], one_step[train_slice], "train_one_step"))
    metrics.update(error_metrics(derivatives[val_slice], derivatives_pred[val_slice], "val_derivative"))
    metrics.update(error_metrics(reduced.states[val_slice], one_step[val_slice], "val_one_step"))
    metrics["overfit_derivative_r2_gap"] = None if metrics["train_derivative_r2"] is None or metrics["val_derivative_r2"] is None else float(metrics["train_derivative_r2"] - metrics["val_derivative_r2"])
    metrics["overfit_one_step_r2_gap"] = None if metrics["train_one_step_r2"] is None or metrics["val_one_step_r2"] is None else float(metrics["train_one_step_r2"] - metrics["val_one_step_r2"])

    return {
        **cfg,
        "latent_pod_energy_capture": float(reduced.metadata["latent_pod_energy_capture"]),
        "frequency": float(freq),
        "library_size": int(theta.shape[1]),
        "n_possible_terms": n_possible,
        "n_nonzero_terms": n_nonzero,
        "sparsity": float(1.0 - n_nonzero / max(n_possible, 1)),
        "derivative_l2_error": float(derivative_error),
        "one_step_l2_error": float(one_step_error),
        "free_integration_success": bool(free_success),
        "free_trajectory_l2_error": None if free_error is None else float(free_error),
        "aic": float(aic),
        "integration_message": message,
        **metrics,
        "dataset": reduced,
        "transform": transform,
        "library": library,
        "Xi": Xi,
        "free_pred": free_pred,
        "one_step_pred": one_step,
        "derivatives_reference": derivatives,
        "derivatives_pred": derivatives_pred,
    }




def rolling_mean(values: np.ndarray, window: int) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    if window <= 1:
        return values
    kernel = np.ones(int(window), dtype=np.float64) / float(window)
    return np.convolve(values, kernel, mode="same")


def persistence_baseline(states: np.ndarray) -> np.ndarray:
    baseline = np.empty_like(states, dtype=np.float64)
    baseline[0] = states[0]
    baseline[1:] = states[:-1]
    return baseline


def period_learning_table(
    times: np.ndarray,
    reference: np.ndarray,
    prediction: np.ndarray,
    train_count: int,
    n_periods: int = 20,
) -> list[dict[str, Any]]:
    times = np.asarray(times, dtype=np.float64)
    reference = np.asarray(reference, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    baseline = persistence_baseline(reference)
    edges = np.linspace(0, len(times), n_periods + 1, dtype=int)
    rows = []

    for period in range(n_periods):
        i0, i1 = int(edges[period]), int(edges[period + 1])
        if i1 <= i0:
            continue
        ref = reference[i0:i1]
        pred = prediction[i0:i1]
        base = baseline[i0:i1]
        rmse_model = float(np.sqrt(np.mean((pred - ref) ** 2)))
        rmse_base = float(np.sqrt(np.mean((base - ref) ** 2)))
        learning_gain = float(1.0 - rmse_model / (rmse_base + 1e-12))
        rows.append({
            "period": period,
            "index_start": i0,
            "index_end": i1 - 1,
            "time_start": float(times[i0]),
            "time_end": float(times[i1 - 1]),
            "split": "train" if i1 <= train_count else "validation" if i0 >= train_count else "train/validation",
            "rmse": rmse_model,
            "mae": float(np.mean(np.abs(pred - ref))),
            "max_abs": float(np.max(np.abs(pred - ref))),
            "rel_l2": relative_l2(ref, pred),
            "r2": r2_score_array(ref, pred),
            "baseline_rmse": rmse_base,
            "learning_gain_vs_persistence": learning_gain,
        })
    return rows


def coordinate_learning_matrix(reference: np.ndarray, prediction: np.ndarray, n_periods: int = 20) -> np.ndarray:
    reference = np.asarray(reference, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    baseline = persistence_baseline(reference)
    edges = np.linspace(0, reference.shape[0], n_periods + 1, dtype=int)
    matrix = np.full((reference.shape[1], n_periods), np.nan, dtype=np.float64)

    for period in range(n_periods):
        i0, i1 = int(edges[period]), int(edges[period + 1])
        if i1 <= i0:
            continue
        for coord in range(reference.shape[1]):
            ref = reference[i0:i1, coord]
            pred = prediction[i0:i1, coord]
            base = baseline[i0:i1, coord]
            rmse_model = float(np.sqrt(np.mean((pred - ref) ** 2)))
            rmse_base = float(np.sqrt(np.mean((base - ref) ** 2)))
            matrix[coord, period] = 1.0 - rmse_model / (rmse_base + 1e-12)
    return matrix


def plot_temporal_learning_analysis(best: dict[str, Any], output_dir: Path) -> dict[str, str]:
    output_dir = Path(output_dir)
    times = np.asarray(best["dataset"].times, dtype=np.float64)
    states = np.asarray(best["dataset"].states, dtype=np.float64)
    train_count = int(best["dataset"].metadata.get("train_count", int(0.8 * len(times))))
    prediction = best["free_pred"] if best["free_pred"] is not None and np.all(np.isfinite(best["free_pred"])) else best["one_step_pred"]
    prediction = np.asarray(prediction, dtype=np.float64)

    err_norm = np.linalg.norm(prediction - states, axis=1) / (np.linalg.norm(states, axis=1) + 1e-12)
    abs_err = np.linalg.norm(prediction - states, axis=1)
    baseline = persistence_baseline(states)
    base_err = np.linalg.norm(baseline - states, axis=1)
    learning_gain_t = 1.0 - abs_err / (base_err + 1e-12)
    window = max(5, len(times) // 50)

    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    axes[0].plot(times, err_norm, color="tab:red", alpha=0.35, linewidth=0.8, label="instantâneo")
    axes[0].plot(times, rolling_mean(err_norm, window), color="tab:red", linewidth=1.8, label="média móvel")
    axes[0].axvline(times[min(train_count, len(times) - 1)], color="black", linestyle="--", linewidth=1.0, label="fim treino")
    axes[0].set_ylabel("erro relativo")
    axes[0].legend(fontsize=8)

    axes[1].plot(times, learning_gain_t, color="tab:blue", alpha=0.35, linewidth=0.8, label="instantâneo")
    axes[1].plot(times, rolling_mean(learning_gain_t, window), color="tab:blue", linewidth=1.8, label="média móvel")
    axes[1].axhline(0.0, color="black", linestyle=":", linewidth=1.0)
    axes[1].axvline(times[min(train_count, len(times) - 1)], color="black", linestyle="--", linewidth=1.0)
    axes[1].set_ylabel("ganho vs persistência")
    axes[1].legend(fontsize=8)

    for coord in range(states.shape[1]):
        coord_err = np.abs(prediction[:, coord] - states[:, coord])
        axes[2].plot(times, rolling_mean(coord_err, window), linewidth=1.2, label=f"q{coord + 1}")
    axes[2].axvline(times[min(train_count, len(times) - 1)], color="black", linestyle="--", linewidth=1.0)
    axes[2].set_xlabel("tempo [s]")
    axes[2].set_ylabel("erro abs. latente")
    axes[2].legend(fontsize=8, ncol=min(states.shape[1], 4))
    fig.tight_layout()
    curves_path = output_dir / "temporal_error_curves.png"
    fig.savefig(curves_path, dpi=180)
    plt.close(fig)

    n_periods = 20
    period_rows = period_learning_table(times, states, prediction, train_count, n_periods=n_periods)
    heatmap = coordinate_learning_matrix(states, prediction, n_periods=n_periods)
    fig, ax = plt.subplots(figsize=(10, 3.2))
    vmax = float(np.nanmax(np.abs(heatmap))) if np.isfinite(heatmap).any() else 1.0
    im = ax.imshow(heatmap, origin="lower", aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_title("Taxa de aprendizado local por período")
    ax.set_xlabel("período temporal")
    ax.set_ylabel("coordenada latente")
    ax.set_yticks(range(states.shape[1]))
    ax.set_yticklabels([f"q{i + 1}" for i in range(states.shape[1])])
    ax.set_xticks(range(n_periods))
    ax.set_xticklabels([str(i + 1) for i in range(n_periods)], fontsize=8)
    split_period = int(round(train_count / max(len(times), 1) * n_periods)) - 0.5
    ax.axvline(split_period, color="black", linestyle="--", linewidth=1.0)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    heatmap_path = output_dir / "temporal_learning_heatmap.png"
    fig.savefig(heatmap_path, dpi=180)
    plt.close(fig)

    csv_path = output_dir / "temporal_error_by_period.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(period_rows[0].keys()))
        writer.writeheader()
        writer.writerows(period_rows)

    json_path = output_dir / "temporal_error_by_period.json"
    json_path.write_text(json.dumps(make_json_ready(period_rows), indent=2), encoding="utf-8")

    return {
        "temporal_error_curves": str(curves_path),
        "temporal_learning_heatmap": str(heatmap_path),
        "temporal_error_csv": str(csv_path),
        "temporal_error_json": str(json_path),
    }

def run_sindy_search(case_dir: Path, output_dir: Path, config: SearchConfig) -> dict[str, Any]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    latent_dataset = load_sae_latents(case_dir)
    rows = []
    print("\nBusca SAE-POD-SINDy")
    print(f"Latentes: {latent_dataset.states.shape} | Fonte: {latent_dataset.source}")

    for modes in config.latent_pod_modes_list:
        for order in config.polynomial_orders:
            for method in config.methods:
                for alpha in config.alpha_values:
                    for smooth in config.smooth_windows:
                        for use_time_poly in config.use_time_poly_options:
                            for use_time_sine in config.use_time_sine_options:
                                for use_state_sine in config.use_state_sine_options:
                                    for integrator in config.integrator_methods:
                                        cfg = dict(
                                            latent_pod_modes=modes,
                                            polynomial_order=order,
                                            method=method,
                                            alpha=alpha,
                                            smooth_window=smooth,
                                            use_time_poly=use_time_poly,
                                            use_time_sine=use_time_sine,
                                            use_state_sine=use_state_sine,
                                            integrator=integrator,
                                        )
                                        item = evaluate_candidate(latent_dataset, cfg)
                                        item["accepted"] = bool(
                                            item["free_integration_success"]
                                            and item["derivative_l2_error"] <= config.derivative_error_limit
                                            and item["free_trajectory_l2_error"] is not None
                                            and item["free_trajectory_l2_error"] <= config.trajectory_error_limit
                                        )
                                        rows.append(item)
                                        status = "OK" if item["accepted"] else "--"
                                        print(
                                            f"[{status}] m={modes} ord={order} {method} a={alpha:.1e} sm={smooth} "
                                            f"tpoly={use_time_poly} tsin={use_time_sine} int={integrator} "
                                            f"der={item['derivative_l2_error']:.4f} free={item['free_trajectory_l2_error']} "
                                            f"nnz={item['n_nonzero_terms']} AIC={item['aic']:.2f}"
                                        )

    accepted = [r for r in rows if r["accepted"]]
    if accepted:
        best = sorted(accepted, key=lambda r: (r["free_trajectory_l2_error"], r["aic"], r["n_nonzero_terms"]))[0]
        reason = "modelo aceito com menor erro livre, AIC e parcimônia"
    else:
        successful = [r for r in rows if r["free_integration_success"]]
        pool = successful if successful else rows
        best = sorted(pool, key=lambda r: ((r["free_trajectory_l2_error"] if r["free_trajectory_l2_error"] is not None else 999), r["aic"], r["n_nonzero_terms"]))[0]
        reason = "nenhum aceito; escolhido menor erro livre disponível"

    metric_fields = [
        "derivative_rmse", "derivative_mae", "derivative_max_abs", "derivative_rel_l2", "derivative_r2",
        "one_step_rmse", "one_step_mae", "one_step_max_abs", "one_step_rel_l2", "one_step_r2",
        "free_rmse", "free_mae", "free_max_abs", "free_rel_l2", "free_r2",
        "train_derivative_rmse", "train_derivative_mae", "train_derivative_max_abs", "train_derivative_rel_l2", "train_derivative_r2",
        "val_derivative_rmse", "val_derivative_mae", "val_derivative_max_abs", "val_derivative_rel_l2", "val_derivative_r2",
        "train_one_step_rmse", "train_one_step_mae", "train_one_step_max_abs", "train_one_step_rel_l2", "train_one_step_r2",
        "val_one_step_rmse", "val_one_step_mae", "val_one_step_max_abs", "val_one_step_rel_l2", "val_one_step_r2",
        "overfit_derivative_r2_gap", "overfit_one_step_r2_gap",
    ]
    fields = [
        "accepted", "latent_pod_modes", "latent_pod_energy_capture", "polynomial_order", "method", "alpha", "smooth_window",
        "use_time_poly", "use_time_sine", "use_state_sine", "integrator", "frequency", "library_size", "n_possible_terms",
        "n_nonzero_terms", "sparsity", "derivative_l2_error", "one_step_l2_error", "free_integration_success",
        "free_trajectory_l2_error", "aic", *metric_fields, "integration_message",
    ]
    csv_path = output_dir / "sindy_search_results.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for r in rows:
            writer.writerow({k: r.get(k) for k in fields})

    diagnostic_files = plot_temporal_learning_analysis(best, output_dir)
    (output_dir / "best_equations.txt").write_text(format_equations(best["Xi"], best["library"].feature_names, best["dataset"].state_symbol), encoding="utf-8")
    np.savez_compressed(
        output_dir / "best_model.npz",
        times=best["dataset"].times,
        states=best["dataset"].states,
        Xi=best["Xi"],
        feature_names=np.asarray(best["library"].feature_names, dtype=object),
        free_pred=np.full_like(best["dataset"].states, np.nan) if best["free_pred"] is None else best["free_pred"],
        one_step_pred=best["one_step_pred"],
        derivatives_reference=best["derivatives_reference"],
        derivatives_pred=best["derivatives_pred"],
        z_mean=best["transform"].z_mean,
        z_std=best["transform"].z_std,
        pod_mean=best["transform"].pod_mean,
        components=best["transform"].components,
        q_mean=best["transform"].q_mean,
        q_std=best["transform"].q_std,
    )

    summary = {
        "architecture": "SAE+POD+SINDy",
        "selection_reason": reason,
        "n_tested": len(rows),
        "n_accepted": len(accepted),
        "source_file": str(latent_dataset.source),
        "best": {k: make_json_ready(v) for k, v in best.items() if k not in {"dataset", "transform", "library", "Xi", "free_pred", "one_step_pred", "derivatives_reference", "derivatives_pred"}},
        "files": {
            "csv": str(csv_path),
            **diagnostic_files,
            "best_equations": str(output_dir / "best_equations.txt"),
            "best_model": str(output_dir / "best_model.npz"),
        },
    }
    (output_dir / "summary.json").write_text(json.dumps(make_json_ready(summary), indent=2), encoding="utf-8")
    print("\nMelhor modelo:")
    print(json.dumps(make_json_ready(summary["best"]), indent=2, ensure_ascii=False))
    return summary


## 6. Configuração da busca

Este módulo **só define os parâmetros**. Ele não executa a busca.

A busca está ativada com:

```python
polynomial_orders=(1,)
```

Para voltar à busca até 3ª ordem, mude a configuração para `polynomial_orders=(1, 2, 3)`.


In [6]:
config = SearchConfig(
    latent_pod_modes_list=(4,),
    polynomial_orders=(1,),
    # Para voltar para 3ª ordem:
    # polynomial_orders=(1, 2, 3),
    methods=("alasso", "tlsa_norm"),
    alpha_values=(1e-3, 1e-2, 1e-1),
    smooth_windows=(21, 51),
    use_time_poly_options=(True,),
    use_time_sine_options=(False, True),
    use_state_sine_options=(False,),
    integrator_methods=("RK45", "Radau"),
    derivative_error_limit=0.95,
    trajectory_error_limit=0.80,
)

run_output = OUTPUT_ROOT / "order1_time_alasso"
print(config)
print(f"Saída: {run_output}")


ArticleSearchConfig(latent_pod_modes_list=(4,), polynomial_orders=(1, 2, 3), methods=('alasso', 'tlsa_norm'), alpha_values=(0.001, 0.01, 0.1), smooth_windows=(21, 51), use_time_poly_options=(True,), use_time_sine_options=(False, True), use_state_sine_options=(False,), integrator_methods=('RK45', 'Radau'), trajectory_error_limit=0.8, derivative_error_limit=0.95)
Saída: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SINDy/resultados/sae_pod_sindy_article_copy/article_order123_time_alasso


## 7. Diagnóstico dos dados latentes

Este módulo carrega os latentes SAE e aplica POD latente apenas para verificar dimensões e energia antes da busca.


In [7]:
latent_dataset = load_sae_latents(CASE_DIR)
reduced_preview, transform_preview = fit_latent_pod(latent_dataset, n_modes=4)

print(f"Latentes SAE: {latent_dataset.states.shape}")
print(f"Tempo: {latent_dataset.times[0]:.6f} -> {latent_dataset.times[-1]:.6f}")
print(f"dt mediano: {latent_dataset.metadata['dt']}")
print(f"POD latente: {reduced_preview.states.shape}")
print(f"Energia POD latente com 4 modos: {reduced_preview.metadata['latent_pod_energy_capture']:.12f}")


Latentes SAE: (1001, 4)
Tempo: 0.000000 -> 1.000000
dt mediano: 0.0009999871253967285
POD latente: (1001, 4)
Energia POD latente com 4 modos: 1.000000000000


## 8. Executar busca SAE-POD-SINDy

Este módulo executa a busca. Ele salva resultados em `run_output`.

Se quiser rodar uma busca rápida, volte ao módulo de configuração e use:

```python
polynomial_orders=(1,)
```


In [8]:
summary = run_sindy_search(CASE_DIR, run_output, config)
summary



Busca SAE-POD-SINDy estilo artigo
Latentes: (1001, 4) | Fonte: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SAE/resultados/latents.npz
[OK] m=4 ord=1 alasso a=1.0e-03 sm=21 tpoly=True tsin=False int=RK45 der=0.8892 free=0.7417408146553051 nnz=27 AIC=-2338.43
[OK] m=4 ord=1 alasso a=1.0e-03 sm=21 tpoly=True tsin=False int=Radau der=0.8892 free=0.7417412313584624 nnz=27 AIC=-2338.43
[OK] m=4 ord=1 alasso a=1.0e-03 sm=21 tpoly=True tsin=True int=RK45 der=0.8811 free=0.5598786395274168 nnz=36 AIC=-4572.92
[OK] m=4 ord=1 alasso a=1.0e-03 sm=21 tpoly=True tsin=True int=Radau der=0.8811 free=0.5598790270463162 nnz=36 AIC=-4572.92
[--] m=4 ord=1 alasso a=1.0e-03 sm=51 tpoly=True tsin=False int=RK45 der=0.8449 free=0.8730762865783424 nnz=32 AIC=-1022.94
[--] m=4 ord=1 alasso a=1.0e-03 sm=51 tpoly=True tsin=False int=Radau der=0.8449 free=0.8730767553344461 nnz=32 AIC=-1022.94
[OK] m=4 ord=1 alasso a=1.0e-03 sm=51 tpoly=Tr

{'architecture': 'SAE+POD+SINDy_article_copy',
 'selection_reason': 'modelo aceito com menor erro livre, AIC e parcimônia',
 'n_tested': 144,
 'n_accepted': 40,
 'source_file': '/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SAE/resultados/latents.npz',
 'best': {'latent_pod_modes': 4,
  'polynomial_order': 1,
  'method': 'alasso',
  'alpha': 0.01,
  'smooth_window': 21,
  'use_time_poly': True,
  'use_time_sine': False,
  'use_state_sine': False,
  'integrator': 'RK45',
  'latent_pod_energy_capture': 0.9999999999999998,
  'frequency': 0.9990138609081209,
  'library_size': 8,
  'n_possible_terms': 32,
  'n_nonzero_terms': 27,
  'sparsity': 0.15625,
  'derivative_l2_error': 0.9026376103848521,
  'one_step_l2_error': 0.030077385625927506,
  'free_integration_success': True,
  'free_trajectory_l2_error': 0.5509056245385466,
  'aic': -4720.303652748018,
  'integration_message': None,
  'accepted': True},
 'files': {'csv

## 9. Métricas do melhor modelo

Este módulo lê os arquivos salvos e imprime métricas para análise.


In [9]:
summary_path = run_output / "summary.json"
summary = json.loads(summary_path.read_text())
best = summary["best"]

print("=== Melhor modelo SAE-POD-SINDy ===")
for key in [
    "latent_pod_modes", "latent_pod_energy_capture", "polynomial_order", "method", "alpha", "smooth_window",
    "use_time_poly", "use_time_sine", "integrator", "library_size", "n_nonzero_terms", "sparsity",
    "derivative_rmse", "derivative_mae", "derivative_rel_l2", "derivative_r2",
    "one_step_rmse", "one_step_mae", "one_step_rel_l2", "one_step_r2",
    "free_rmse", "free_mae", "free_rel_l2", "free_r2",
    "train_one_step_r2", "val_one_step_r2", "overfit_one_step_r2_gap",
    "train_derivative_r2", "val_derivative_r2", "overfit_derivative_r2_gap",
    "free_integration_success", "aic",
]:
    print(f"{key:32s}: {best.get(key)}")

model = np.load(run_output / "best_model.npz", allow_pickle=True)
states = model["states"]
free_pred = model["free_pred"]
one_step_pred = model["one_step_pred"]
der_ref = model["derivatives_reference"]
der_pred = model["derivatives_pred"]

print("\n=== Erros recalculados ===")
print(f"Derivadas L2/R²       : {relative_l2(der_ref, der_pred):.6f} / {r2_score_array(der_ref, der_pred):.6f}")
print(f"Um passo L2/R²        : {relative_l2(states, one_step_pred):.6f} / {r2_score_array(states, one_step_pred):.6f}")
if np.isfinite(free_pred).all():
    print(f"Integração livre L2/R²: {relative_l2(states, free_pred):.6f} / {r2_score_array(states, free_pred):.6f}")
    per_coord = np.linalg.norm(states - free_pred, axis=0) / (np.linalg.norm(states, axis=0) + 1e-12)
    print(f"Erro livre por coordenada: {per_coord}")
else:
    print("Integração livre: não disponível ou falhou")

print("\nArquivos:")
print(json.dumps(summary["files"], indent=2, ensure_ascii=False))


=== Melhor modelo SAE-POD-SINDy estilo artigo ===
latent_pod_modes              : 4
latent_pod_energy_capture     : 0.9999999999999998
polynomial_order              : 1
method                        : alasso
alpha                         : 0.01
smooth_window                 : 21
use_time_poly                 : True
use_time_sine                 : False
integrator                    : RK45
library_size                  : 8
n_nonzero_terms               : 27
sparsity                      : 0.15625
derivative_l2_error           : 0.9026376103848521
one_step_l2_error             : 0.030077385625927506
free_integration_success      : True
free_trajectory_l2_error      : 0.5509056245385466
aic                           : -4720.303652748018

=== Erros recalculados ===
Derivadas L2       : 0.902638
Um passo L2        : 0.030077
Integração livre L2: 0.550906
Erro livre por coordenada: [0.50805004 0.52276096 0.49375796 0.66241772]

Arquivos:
{
  "csv": "/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubun

## 10. Funções para reconstrução de imagens físicas

Este módulo define funções para transformar:

```text
coeficientes previstos pelas equações -> latentes SAE -> decoder -> campos físicos
```

As imagens comparadas são:

```text
campo real | campo reconstruído pelas equações | erro
```


In [10]:
def inverse_pod_to_latents(q_scaled: np.ndarray, model_npz: Any) -> np.ndarray:
    q_scaled = np.atleast_2d(np.asarray(q_scaled, dtype=np.float64))
    q = q_scaled * model_npz["q_std"][None, :] + model_npz["q_mean"][None, :]
    z_norm = q @ model_npz["components"] + model_npz["pod_mean"][None, :]
    z = z_norm * model_npz["z_std"][None, :] + model_npz["z_mean"][None, :]
    return z


def build_sae_decoder(case_dir: Path):
    import tensorflow as tf

    results_dir = Path(case_dir) / "SAE" / "resultados"
    model_path = results_dir / "sae_dense_best.keras"
    if not model_path.exists():
        model_path = results_dir / "sae_dense.keras"
    if not model_path.exists():
        raise FileNotFoundError(f"Modelo SAE não encontrado em {results_dir}")

    norm_path = results_dir / "sae_dense_norm.npz"
    if not norm_path.exists():
        raise FileNotFoundError(f"Normalização SAE não encontrada: {norm_path}")

    autoencoder = tf.keras.models.load_model(model_path, compile=False)
    norm_data = np.load(norm_path, allow_pickle=True)
    mu = np.asarray(norm_data["mu"], dtype=np.float64).reshape(-1)
    sig = np.asarray(norm_data["sig"], dtype=np.float64).reshape(-1)
    channel_names = [str(x) for x in np.asarray(norm_data["channel_names"]).ravel()] if "channel_names" in norm_data.files else ["Ux", "Uy"]
    n_channels = len(channel_names)
    pixels = mu.size // n_channels
    side = int(np.sqrt(pixels))
    if side * side != pixels:
        raise ValueError(f"Não foi possível inferir grade quadrada: mu.size={mu.size}, canais={n_channels}")

    layer_names = [layer.name for layer in autoencoder.layers]
    if "latent" not in layer_names:
        raise ValueError(f"Camada 'latent' não encontrada. Camadas: {layer_names}")
    decoder_layer_names = layer_names[layer_names.index("latent") + 1:]

    latent_input = tf.keras.Input(shape=(autoencoder.get_layer("latent").output.shape[-1],), name="latent_input")
    x = latent_input
    for layer_name in decoder_layer_names:
        x = autoencoder.get_layer(layer_name)(x)
    decoder = tf.keras.Model(latent_input, x, name="sae_decoder")

    def decode(z: np.ndarray) -> np.ndarray:
        z = np.atleast_2d(np.asarray(z, dtype=np.float32))
        decoded_norm = decoder.predict(z, verbose=0)
        decoded = decoded_norm * sig[None, :] + mu[None, :]
        return decoded.reshape((-1, side, side, n_channels)).astype(np.float64)

    return decode, channel_names


def field_r2(reference: np.ndarray, estimate: np.ndarray) -> float:
    ss_res = float(np.sum((reference - estimate) ** 2))
    ss_tot = float(np.sum((reference - np.mean(reference)) ** 2))
    return float(1.0 - ss_res / (ss_tot + 1e-12))


def decoded_frame_to_panels(frame: np.ndarray, channel_names: list[str]) -> dict[str, np.ndarray]:
    channel_index = {name: i for i, name in enumerate(channel_names)}
    fields = {}
    if "Ux" in channel_index and "Uy" in channel_index:
        ux = frame[..., channel_index["Ux"]]
        uy = frame[..., channel_index["Uy"]]
        fields["|U|"] = np.hypot(ux, uy)
    if "alpha.water" in channel_index:
        fields["alpha.water"] = frame[..., channel_index["alpha.water"]]
    elif len(channel_names) > 2:
        fields[channel_names[2]] = frame[..., 2]
    if not fields:
        fields[channel_names[0]] = frame[..., 0]
    return fields


def compute_field_comparison_scales(records: list[dict[str, Any]]) -> dict[str, dict[str, float]]:
    names = sorted({name for record in records for name in record["ref_fields"]})
    scales = {}
    for name in names:
        values = []
        errors = []
        for record in records:
            if name not in record["ref_fields"] or name not in record["pred_fields"]:
                continue
            ref = record["ref_fields"][name]
            pred = record["pred_fields"][name]
            values.extend([float(np.nanmin(ref)), float(np.nanmax(ref)), float(np.nanmin(pred)), float(np.nanmax(pred))])
            errors.append(float(np.nanmax(np.abs(pred - ref))))
        scales[name] = {
            "vmin": float(np.nanmin(values)),
            "vmax": float(np.nanmax(values)),
            "err_lim": float(np.nanmax(errors)) if errors else 1e-12,
        }
    return scales


def save_field_comparison(
    ref_frame: np.ndarray,
    pred_frame: np.ndarray,
    channel_names: list[str],
    output_path: Path,
    time_s: float,
    scales: dict[str, dict[str, float]],
) -> dict[str, float]:
    force_matplotlib_agg()
    ref_fields = decoded_frame_to_panels(ref_frame, channel_names)
    pred_fields = decoded_frame_to_panels(pred_frame, channel_names)
    field_names = [name for name in ref_fields if name in pred_fields]

    fig, axes = plt.subplots(len(field_names), 3, figsize=(12, 4 * len(field_names)), squeeze=False)
    metrics = {}
    time_label = f"t={time_s:.2f} s"

    for row, name in enumerate(field_names):
        ref = np.asarray(ref_fields[name], dtype=np.float64)
        pred = np.asarray(pred_fields[name], dtype=np.float64)
        err = pred - ref
        rel_l2 = relative_l2(ref, pred)
        r2 = field_r2(ref, pred)
        metrics[f"{name}_rel_l2"] = rel_l2
        metrics[f"{name}_r2"] = r2

        scale = scales[name]
        vmin = scale["vmin"]
        vmax = scale["vmax"]
        err_lim = max(scale["err_lim"], 1e-12)
        panels = (
            (ref, f"{name} SAE ref. {time_label}", "viridis", vmin, vmax),
            (pred, f"{name} SINDy/SAE {time_label}", "viridis", vmin, vmax),
            (err, f"{name} erro físico {time_label}\nL2rel={rel_l2:.3g} | R²={r2:.3g}", "coolwarm", -err_lim, err_lim),
        )

        for col, (img, panel_title, cmap, cmin, cmax) in enumerate(panels):
            ax = axes[row, col]
            im = ax.imshow(img, origin="lower", cmap=cmap, vmin=cmin, vmax=cmax, interpolation="bicubic", aspect="equal")
            ax.set_title(panel_title, fontsize=10)
            ax.set_box_aspect(1)
            ax.axis("off")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.close(fig)
    return metrics


## 11. Gerar imagens em instantes específicos

Este módulo gera imagens para análise visual em tempos definidos.

Ele salva, para cada tempo:

```text
real/SAE | reconstruído pelas equações | erro
```


In [11]:
requested_times = (0.0, 0.25, 0.50, 0.75, 1.0)
image_output_dir = run_output / "field_comparisons_exact_times"
image_output_dir.mkdir(parents=True, exist_ok=True)

summary = json.loads((run_output / "summary.json").read_text())
model = np.load(run_output / "best_model.npz", allow_pickle=True)
latent_dataset = load_sae_latents(CASE_DIR)
decode, channel_names = build_sae_decoder(CASE_DIR)

free_pred = model["free_pred"]
if not np.isfinite(free_pred).all():
    print("Integração livre indisponível; usando previsão de um passo.")
    q_pred_all = model["one_step_pred"]
    trajectory_label = "one_step"
else:
    q_pred_all = free_pred
    trajectory_label = "free_integration"

z_pred_all = inverse_pod_to_latents(q_pred_all, model)
records = []

for target in requested_times:
    idx = int(np.argmin(np.abs(latent_dataset.times - target)))
    actual = float(latent_dataset.times[idx])
    ref_frame = decode(latent_dataset.states[idx:idx + 1])[0]
    pred_frame = decode(z_pred_all[idx:idx + 1])[0]
    records.append({
        "requested_time": float(target),
        "actual_time": actual,
        "index": int(idx),
        "ref_frame": ref_frame,
        "pred_frame": pred_frame,
        "ref_fields": decoded_frame_to_panels(ref_frame, channel_names),
        "pred_fields": decoded_frame_to_panels(pred_frame, channel_names),
    })

scales = compute_field_comparison_scales(records)
report = []

for record in records:
    out_file = image_output_dir / f"time_{record['requested_time']:04.2f}s.png"
    metrics = save_field_comparison(
        record["ref_frame"],
        record["pred_frame"],
        channel_names,
        out_file,
        time_s=record["actual_time"],
        scales=scales,
    )
    item = {
        "requested_time": record["requested_time"],
        "actual_time": record["actual_time"],
        "index": record["index"],
        "trajectory": trajectory_label,
        "file": str(out_file),
        **metrics,
    }
    report.append(item)
    print(item)

(image_output_dir / "report.json").write_text(json.dumps(make_json_ready(report), indent=2), encoding="utf-8")
(image_output_dir / "plot_scales.json").write_text(json.dumps(make_json_ready(scales), indent=2), encoding="utf-8")
print(f"Imagens salvas em: {image_output_dir}")


2026-09-27 14:00:40.152910: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


{'requested_time': 0.0, 'actual_time': 0.0, 'index': 0, 'file': '/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SINDy/resultados/sae_pod_sindy_article_copy/article_order123_time_alasso/field_comparisons_exact_times/time_0.00s.png', '|U|_rel_l2': 0.0, 'alpha.water_rel_l2': 0.0}
{'requested_time': 0.25, 'actual_time': 0.25, 'index': 250, 'file': '/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SINDy/resultados/sae_pod_sindy_article_copy/article_order123_time_alasso/field_comparisons_exact_times/time_0.25s.png', '|U|_rel_l2': 0.2887505078773497, 'alpha.water_rel_l2': 0.2140748378369492}
{'requested_time': 0.5, 'actual_time': 0.5, 'index': 500, 'file': '/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SINDy/resultados/sae_pod_sindy_article_copy/article_order123_time_alasso/field_comp

## 12. Resumo JSON final


In [ ]:
print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "architecture": "SAE+POD+SINDy_article_copy",
  "selection_reason": "modelo aceito com menor erro livre, AIC e parcimônia",
  "n_tested": 144,
  "n_accepted": 40,
  "source_file": "/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SAE/resultados/latents.npz",
  "best": {
    "latent_pod_modes": 4,
    "polynomial_order": 1,
    "method": "alasso",
    "alpha": 0.01,
    "smooth_window": 21,
    "use_time_poly": true,
    "use_time_sine": false,
    "use_state_sine": false,
    "integrator": "RK45",
    "latent_pod_energy_capture": 0.9999999999999998,
    "frequency": 0.9990138609081209,
    "library_size": 8,
    "n_possible_terms": 32,
    "n_nonzero_terms": 27,
    "sparsity": 0.15625,
    "derivative_l2_error": 0.9026376103848521,
    "one_step_l2_error": 0.030077385625927506,
    "free_integration_success": true,
    "free_trajectory_l2_error": 0.5509056245385466,
    "aic": -4720.303652748018,
    "integration

In [23]:
@dataclass
class OpenFOAMGridReference:
    U: np.ndarray
    alpha: np.ndarray
    x: np.ndarray
    y: np.ndarray
    grid_x: np.ndarray
    grid_y: np.ndarray

    @property
    def n_snapshots(self) -> int:
        return int(self.U.shape[0])

    def interpolate(self, values: np.ndarray) -> np.ndarray:
        values = np.asarray(values, dtype=np.float64).ravel()
        valid = np.isfinite(self.x) & np.isfinite(self.y) & np.isfinite(values)
        if valid.sum() < 3:
            raise ValueError("Poucos pontos válidos para interpolação.")
        points = np.column_stack((self.x[valid], self.y[valid]))
        field = griddata(points, values[valid], (self.grid_x, self.grid_y), method="linear")
        if np.isnan(field).any():
            nearest = griddata(points, values[valid], (self.grid_x, self.grid_y), method="nearest")
            field = np.where(np.isnan(field), nearest, field)
        return np.asarray(field, dtype=np.float64)

    def fields(self, index: int) -> dict[str, np.ndarray]:
        velocity = np.asarray(self.U[index], dtype=np.float64)
        alpha = np.squeeze(np.asarray(self.alpha[index], dtype=np.float64))
        if velocity.ndim != 2 or velocity.shape[1] < 2:
            raise ValueError(f"Snapshot U inválido: {velocity.shape}")

        ux = self.interpolate(velocity[:, 0])
        uy = self.interpolate(velocity[:, 1])
        return {
            "Ux": ux,
            "Uy": uy,
            "|U|": np.hypot(ux, uy),
            "alpha.water": self.interpolate(alpha),
        }


def build_openfoam_grid_reference(case_dir: Path, grid_size: int = 128) -> OpenFOAMGridReference:
    cache_dir = Path(case_dir) / "rom_melhorias" / "cache"
    paths = {
        "U": cache_dir / "snapshots_U.npy",
        "alpha": cache_dir / "snapshots_alpha.water.npy",
        "centers": cache_dir / "cell_centers.npy",
    }
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("Arquivos OpenFOAM ausentes:\n" + "\n".join(missing))

    U = np.load(paths["U"], mmap_mode="r")
    alpha = np.load(paths["alpha"], mmap_mode="r")
    centers = np.asarray(np.load(paths["centers"]), dtype=np.float64)
    if centers.ndim != 2 or centers.shape[1] < 2:
        raise ValueError(f"cell_centers.npy inválido: {centers.shape}")

    x = centers[:, 0]
    y = centers[:, 1]
    grid_x, grid_y = np.meshgrid(
        np.linspace(float(x.min()), float(x.max()), grid_size),
        np.linspace(float(y.min()), float(y.max()), grid_size),
    )
    return OpenFOAMGridReference(U, alpha, x, y, grid_x, grid_y)


def sae_frame_to_fields(frame: np.ndarray, channel_names: list[str]) -> dict[str, np.ndarray]:
    frame = np.asarray(frame, dtype=np.float64)
    n_channels = len(channel_names)
    if frame.ndim != 3:
        raise ValueError(f"Frame SAE inválido: {frame.shape}")
    if frame.shape[0] == n_channels and frame.shape[-1] != n_channels:
        frame = np.moveaxis(frame, 0, -1)
    if frame.shape[-1] != n_channels:
        raise ValueError(f"Canais incompatíveis: frame={frame.shape}, nomes={channel_names}")

    idx = {name: i for i, name in enumerate(channel_names)}
    fields = {}
    for name in ("Ux", "Uy", "alpha.water"):
        if name in idx:
            fields[name] = frame[..., idx[name]]
    if "Ux" in fields and "Uy" in fields:
        fields["|U|"] = np.hypot(fields["Ux"], fields["Uy"])
    return fields


def resize_field(field: np.ndarray, target_size: int = 128, order: int = 3) -> np.ndarray:
    field = np.asarray(field, dtype=np.float64)
    if field.ndim != 2:
        raise ValueError(f"Campo 2D esperado, recebido {field.shape}")
    if field.shape == (target_size, target_size):
        return field

    factors = (target_size / field.shape[0], target_size / field.shape[1])
    resized = zoom(field, factors, order=order)[:target_size, :target_size]
    if resized.shape != (target_size, target_size):
        padded = np.empty((target_size, target_size), dtype=np.float64)
        padded[:] = np.nan
        padded[:resized.shape[0], :resized.shape[1]] = resized
        nearest = zoom(field, factors, order=0)[:target_size, :target_size]
        resized = np.where(np.isnan(padded), nearest, padded)

    return np.clip(resized, float(np.nanmin(field)), float(np.nanmax(field)))


def resize_fields(fields: dict[str, np.ndarray], target_size: int = 128) -> dict[str, np.ndarray]:
    resized = {
        name: resize_field(field, target_size)
        for name, field in fields.items()
        if name != "|U|"
    }
    if "Ux" in resized and "Uy" in resized:
        resized["|U|"] = np.hypot(resized["Ux"], resized["Uy"])
    elif "|U|" in fields:
        resized["|U|"] = resize_field(fields["|U|"], target_size)
    return resized


def coefficient_of_determination(reference: np.ndarray, estimate: np.ndarray) -> float:
    reference = np.asarray(reference, dtype=np.float64)
    estimate = np.asarray(estimate, dtype=np.float64)
    ss_res = float(np.sum((reference - estimate) ** 2))
    ss_tot = float(np.sum((reference - np.mean(reference)) ** 2))
    return float(1.0 - ss_res / (ss_tot + 1e-12))


def plot_openfoam_vs_sindy(
    openfoam_fields: dict[str, np.ndarray],
    sindy_fields: dict[str, np.ndarray],
    output_path: Path,
    title: str,
    time_s: float,
    scales: dict[str, dict[str, float]],
    field_names: tuple[str, ...] = ("|U|", "alpha.water"),
) -> dict[str, float]:
    force_matplotlib_agg()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    metrics = {}
    fig, axes = plt.subplots(len(field_names), 3, figsize=(12, 4 * len(field_names)), squeeze=False)

    for row, name in enumerate(field_names):
        if name not in openfoam_fields or name not in sindy_fields:
            print(f"Campo ausente na comparação: {name}")
            continue

        ref = np.asarray(openfoam_fields[name], dtype=np.float64)
        pred = np.asarray(sindy_fields[name], dtype=np.float64)
        if ref.shape != pred.shape:
            raise ValueError(f"Shape incompatível em {name}: OpenFOAM={ref.shape}, SINDy={pred.shape}")

        err = pred - ref
        rel_l2 = relative_l2(ref, pred)
        r2 = coefficient_of_determination(ref, pred)
        metrics[f"{name}_rel_l2"] = rel_l2
        metrics[f"{name}_r2"] = r2

        field_scale = scales[name]
        vmin = field_scale["vmin"]
        vmax = field_scale["vmax"]
        err_lim = max(field_scale["err_lim"], 1e-12)
        field_label = f"valor físico [{vmin:.3g}, {vmax:.3g}]"
        err_label = f"SINDy - OpenFOAM [{-err_lim:.3g}, {err_lim:.3g}]"
        time_label = f"t={time_s:.2f} s"

        panels = (
            (ref, f"{name} OpenFOAM {time_label}", "viridis", vmin, vmax, field_label),
            (pred, f"{name} SINDy/SAE {time_label}", "viridis", vmin, vmax, field_label),
            (err, f"{name} erro físico {time_label}\nL2rel={rel_l2:.3g} | R²={r2:.3g}", "coolwarm", -err_lim, err_lim, err_label),
        )

        for col, (image, panel_title, cmap, cmin, cmax, label) in enumerate(panels):
            ax = axes[row, col]
            im = ax.imshow(
                image,
                origin="lower",
                cmap=cmap,
                vmin=cmin,
                vmax=cmax,
                interpolation="bicubic",
                aspect="equal",
            )
            ax.set_title(panel_title, fontsize=10)
            ax.set_box_aspect(1)
            ax.axis("off")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    if title:
        fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.close(fig)
    return metrics

def select_sindy_trajectory(model: Any) -> tuple[np.ndarray, str]:
    if "free_pred" in model and np.isfinite(model["free_pred"]).all():
        return np.asarray(model["free_pred"], dtype=np.float64), "free_integration"
    if "one_step_pred" in model:
        return np.asarray(model["one_step_pred"], dtype=np.float64), "one_step"
    raise KeyError("best_model.npz não possui free_pred nem one_step_pred.")


def temporal_diagnostics(reference: OpenFOAMGridReference, index: int, sindy_step: int, fields: dict[str, np.ndarray]) -> dict[str, float]:
    values = {}
    for neighbor in (index - sindy_step, index + sindy_step):
        if not 0 <= neighbor < reference.n_snapshots:
            continue
        neighbor_fields = reference.fields(neighbor)
        for name in ("|U|", "alpha.water"):
            key = f"{name}_openfoam_1_sindy_step_rel_l2"
            error = relative_l2(fields[name], neighbor_fields[name])
            values[key] = min(values.get(key, float("inf")), float(error))
    return values


def compute_plot_scales(records: list[dict[str, Any]], field_names: tuple[str, ...]) -> dict[str, dict[str, float]]:
    scales = {}
    for name in field_names:
        field_values = []
        error_values = []
        for record in records:
            if name not in record["openfoam_fields"] or name not in record["sindy_fields"]:
                continue
            ref = np.asarray(record["openfoam_fields"][name], dtype=np.float64)
            pred = np.asarray(record["sindy_fields"][name], dtype=np.float64)
            field_values.extend([float(np.nanmin(ref)), float(np.nanmax(ref)), float(np.nanmin(pred)), float(np.nanmax(pred))])
            error_values.append(float(np.nanmax(np.abs(pred - ref))))
        scales[name] = {
            "vmin": float(np.nanmin(field_values)),
            "vmax": float(np.nanmax(field_values)),
            "err_lim": float(np.nanmax(error_values)) if error_values else 1e-12,
        }
    return scales


def generate_openfoam_vs_sindy_images(
    case_dir: Path,
    run_output: Path,
    requested_times: tuple[float, ...] = (0.0, 0.25, 0.5, 0.75, 1.0),
    grid_size: int = 128,
):
    case_dir = Path(case_dir)
    run_output = Path(run_output)
    output_dir = run_output / "openfoam_raw_vs_sindy"
    output_dir.mkdir(parents=True, exist_ok=True)

    model_path = run_output / "best_model.npz"
    if not model_path.exists():
        raise FileNotFoundError(f"Modelo não encontrado: {model_path}")

    model = np.load(model_path, allow_pickle=True)
    latent_dataset = load_sae_latents(case_dir)
    times_sindy = np.asarray(latent_dataset.times, dtype=np.float64)
    decode, channel_names = build_sae_decoder(case_dir)
    q_pred, trajectory_label = select_sindy_trajectory(model)
    z_pred = inverse_pod_to_latents(q_pred, model)
    reference = build_openfoam_grid_reference(case_dir, grid_size)

    print("\n=== Comparação OpenFOAM bruto vs SINDy/SAE ===")
    print(f"OpenFOAM snapshots : {reference.n_snapshots}")
    print(f"SINDy/SAE snapshots: {len(z_pred)}")
    print(f"Grade              : {grid_size} x {grid_size}")
    print(f"Trajetória         : {trajectory_label}\n")

    field_names = ("|U|", "alpha.water")
    sindy_step_raw = max(1, int(round((reference.n_snapshots - 1) / max(len(z_pred) - 1, 1))))
    records = []

    for fraction in requested_times:
        fraction = float(fraction)
        if not 0.0 <= fraction <= 1.0:
            raise ValueError(f"Fração inválida: {fraction}")

        sindy_index = int(round(fraction * (len(z_pred) - 1)))
        sindy_index = max(0, min(sindy_index, len(z_pred) - 1))
        sindy_time = float(times_sindy[sindy_index]) if sindy_index < len(times_sindy) else fraction

        openfoam_index = int(round(sindy_time * (reference.n_snapshots - 1)))
        openfoam_index = max(0, min(openfoam_index, reference.n_snapshots - 1))
        openfoam_time = openfoam_index / max(reference.n_snapshots - 1, 1)
        time_mismatch = abs(openfoam_time - sindy_time)

        openfoam_fields = reference.fields(openfoam_index)
        decoded = decode(z_pred[sindy_index:sindy_index + 1])[0]
        sindy_fields = resize_fields(sae_frame_to_fields(decoded, channel_names), grid_size)
        temporal = temporal_diagnostics(reference, openfoam_index, sindy_step_raw, openfoam_fields)

        records.append({
            "fraction": fraction,
            "openfoam_index": int(openfoam_index),
            "openfoam_time_assumed": float(openfoam_time),
            "sindy_index": int(sindy_index),
            "sindy_time": float(sindy_time),
            "time_mismatch": float(time_mismatch),
            "openfoam_fields": openfoam_fields,
            "sindy_fields": sindy_fields,
            "temporal": temporal,
        })

    scales = compute_plot_scales(records, field_names)
    report = []

    for record in records:
        fraction = record["fraction"]
        output_file = output_dir / f"fraction_{fraction:04.2f}.png"
        title = ""
        metrics = plot_openfoam_vs_sindy(
            record["openfoam_fields"],
            record["sindy_fields"],
            output_file,
            title=title,
            time_s=record["sindy_time"],
            scales=scales,
            field_names=field_names,
        )
        item = {
            "fraction": fraction,
            "openfoam_index": record["openfoam_index"],
            "openfoam_time_assumed": record["openfoam_time_assumed"],
            "sindy_index": record["sindy_index"],
            "sindy_time": record["sindy_time"],
            "time_mismatch": record["time_mismatch"],
            "grid_size": int(grid_size),
            "trajectory": trajectory_label,
            "file": str(output_file),
            **metrics,
            **record["temporal"],
        }
        report.append(item)
        print(item)

    report_path = output_dir / "report.json"
    report_path.write_text(json.dumps(make_json_ready(report), indent=2), encoding="utf-8")
    (output_dir / "plot_scales.json").write_text(json.dumps(make_json_ready(scales), indent=2), encoding="utf-8")
    print(f"\nImagens salvas em: {output_dir}")
    return report


openfoam_vs_sindy_report = generate_openfoam_vs_sindy_images(
    case_dir=CASE_DIR,
    run_output=run_output,
    requested_times=(0.0, 0.25, 0.5, 0.75, 1.0),
    grid_size=128,
)

openfoam_vs_sindy_report



=== Alinhamento temporal usado na comparação ===
Snapshots OpenFOAM bruto : 10001
Snapshots SINDy/SAE      : 1001
Tempos SAE disponíveis   : 1001
Usando comparação por fração da trajetória.

{'fraction': 0.0, 'openfoam_index': 0, 'sindy_index': 0, 'sindy_time': 0.0, 'trajectory': 'free_integration', 'file': '/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SINDy/resultados/sae_pod_sindy_article_copy/article_order123_time_alasso/openfoam_raw_vs_sindy/fraction_0.00.png', '|U|_rel_l2': 8041700135419.93, 'alpha.water_rel_l2': 0.4675331854102285}
{'fraction': 0.25, 'openfoam_index': 2500, 'sindy_index': 250, 'sindy_time': 0.25, 'trajectory': 'free_integration', 'file': '/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SINDy/resultados/sae_pod_sindy_article_copy/article_order123_time_alasso/openfoam_raw_vs_sindy/fraction_0.25.png', '|U|_rel_l2': 0.3403363

[{'fraction': 0.0,
  'openfoam_index': 0,
  'sindy_index': 0,
  'sindy_time': 0.0,
  'trajectory': 'free_integration',
  'file': '/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SINDy/resultados/sae_pod_sindy_article_copy/article_order123_time_alasso/openfoam_raw_vs_sindy/fraction_0.00.png',
  '|U|_rel_l2': 8041700135419.93,
  'alpha.water_rel_l2': 0.4675331854102285},
 {'fraction': 0.25,
  'openfoam_index': 2500,
  'sindy_index': 250,
  'sindy_time': 0.25,
  'trajectory': 'free_integration',
  'file': '/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar(URANS)10.000/Codigos(damBreakLaminar10.00(URANS))/SINDy/resultados/sae_pod_sindy_article_copy/article_order123_time_alasso/openfoam_raw_vs_sindy/fraction_0.25.png',
  '|U|_rel_l2': 0.3403363096910032,
  'alpha.water_rel_l2': 0.21854338404511456},
 {'fraction': 0.5,
  'openfoam_index': 5000,
  'sindy_index': 500,
  'sindy_time': 0.5,
  'trajectory': 'f